# Experiment 9: Perceptron vs Multilayer Perceptron (A/B Experiment) with Hyperparameter Tuning
This standalone notebook compares a Single-Layer Perceptron (PLA) implemented from scratch with step activation against a Tuned Multilayer Perceptron (MLP) with backpropagation on the 62-class English Handwritten Characters dataset.

In [1]:
import os
import matplotlib
matplotlib.use('Agg') # Strictly headless - non-interfering, zero GUI popups

def resolve_path(rel_path):
    """Dynamically resolves datasets whether run from repo root or Ex subfolder."""
    for prefix in ['', '../', '../../']:
        cand = os.path.join(prefix, rel_path)
        if os.path.exists(cand):
            return cand
    return rel_path

def resolve_out(rel_path):
    """Avoids nested directories if running from within Ex9."""
    if os.path.basename(os.getcwd()) == 'Ex9':
        if rel_path.startswith('Ex9/'):
            return rel_path[len('Ex9/'):]
    return rel_path

import pandas as pd
import numpy as np
import time
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs(resolve_out('Ex9'), exist_ok=True)

In [2]:
class SingleLayerPLA:
    def __init__(self, n_classes, lr=0.01, max_epochs=30):
        self.n_classes = n_classes
        self.lr = lr
        self.max_epochs = max_epochs
        self.weights = None
        self.biases = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros((self.n_classes, n_features))
        self.biases = np.zeros(self.n_classes)
        y_bin = label_binarize(y, classes=range(self.n_classes))
        for _ in range(self.max_epochs):
            for i in range(n_samples):
                xi = X[i]
                for c in range(self.n_classes):
                    pred = 1 if (np.dot(self.weights[c], xi) + self.biases[c]) >= 0 else 0
                    err = y_bin[i, c] - pred
                    if err != 0:
                        self.weights[c] += self.lr * err * xi
                        self.biases[c] += self.lr * err
        return self

    def predict(self, X):
        return np.argmax(np.dot(X, self.weights.T) + self.biases, axis=1)

In [3]:
def run_experiment_9(data_dir="Datasets/English_Characters", img_size=(28, 28)):
    print("="*60)
    print("=== LAUNCHING EXPERIMENT 9: PLA vs MLP A/B PIPELINE ===")
    print("="*60)
    
    dir_path = resolve_path(data_dir)
    csv_path = os.path.join(dir_path, 'english.csv')
    df = pd.read_csv(csv_path)
    
    images, labels = [], []
    for _, r in df.iterrows():
        fp = os.path.join(dir_path, r['image'])
        if os.path.exists(fp):
            with Image.open(fp) as img:
                arr = np.array(img.convert('L').resize(img_size), dtype=np.float32) / 255.0
                images.append(arr.flatten())
                labels.append(str(r['label']))
                
    X = np.array(images)
    le = LabelEncoder()
    y = le.fit_transform(labels)
    n_classes = len(le.classes_)
    
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    t0 = time.time()
    pla = SingleLayerPLA(n_classes=n_classes, lr=0.01, max_epochs=30)
    pla.fit(X_tr, y_tr)
    pla_time = time.time() - t0
    pla_preds = pla.predict(X_te)
    
    t0 = time.time()
    mlp = MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
                        learning_rate_init=0.001, max_iter=40, random_state=42)
    mlp.fit(X_tr, y_tr)
    mlp_time = time.time() - t0
    mlp_preds = mlp.predict(X_te)
    
    ab_comparison = [
        {
            'Model': 'Single-Layer Perceptron (PLA)',
            'Architecture': 'Single Layer (62 Step Units)',
            'Accuracy (%)': round(accuracy_score(y_te, pla_preds) * 100, 2),
            'Precision (%)': round(precision_score(y_te, pla_preds, average='weighted', zero_division=0) * 100, 2),
            'Recall (%)': round(recall_score(y_te, pla_preds, average='weighted', zero_division=0) * 100, 2),
            'F1-Score (%)': round(f1_score(y_te, pla_preds, average='weighted', zero_division=0) * 100, 2),
            'Training Time (s)': round(pla_time, 2)
        },
        {
            'Model': 'Tuned Multilayer Perceptron (MLP)',
            'Architecture': 'Input(784) -> (128, 64) -> Output(62)',
            'Accuracy (%)': round(accuracy_score(y_te, mlp_preds) * 100, 2),
            'Precision (%)': round(precision_score(y_te, mlp_preds, average='weighted', zero_division=0) * 100, 2),
            'Recall (%)': round(recall_score(y_te, mlp_preds, average='weighted', zero_division=0) * 100, 2),
            'F1-Score (%)': round(f1_score(y_te, mlp_preds, average='weighted', zero_division=0) * 100, 2),
            'Training Time (s)': round(mlp_time, 2)
        }
    ]
    
    print("\n=== EXPERIMENT 9 PIPELINE COMPLETE ===")
    return {'ab_comparison': ab_comparison}

In [4]:
# Master Execution Cell
ex9_output = run_experiment_9()
df_ab = pd.DataFrame(ex9_output['ab_comparison']).set_index('Model')
delta = pd.DataFrame([{
    'Architecture': 'MLP Capacity Gain',
    'Accuracy (%)': df_ab.loc['Tuned Multilayer Perceptron (MLP)', 'Accuracy (%)'] - df_ab.loc['Single-Layer Perceptron (PLA)', 'Accuracy (%)'],
    'Precision (%)': df_ab.loc['Tuned Multilayer Perceptron (MLP)', 'Precision (%)'] - df_ab.loc['Single-Layer Perceptron (PLA)', 'Precision (%)'],
    'Recall (%)': df_ab.loc['Tuned Multilayer Perceptron (MLP)', 'Recall (%)'] - df_ab.loc['Single-Layer Perceptron (PLA)', 'Recall (%)'],
    'F1-Score (%)': df_ab.loc['Tuned Multilayer Perceptron (MLP)', 'F1-Score (%)'] - df_ab.loc['Single-Layer Perceptron (PLA)', 'F1-Score (%)'],
    'Training Time (s)': df_ab.loc['Tuned Multilayer Perceptron (MLP)', 'Training Time (s)'] - df_ab.loc['Single-Layer Perceptron (PLA)', 'Training Time (s)']
}], index=['Gain (Delta)'])
display(pd.concat([df_ab, delta]).style.format({
    'Accuracy (%)': '{:+.2f}%', 'Precision (%)': '{:+.2f}%',
    'Recall (%)': '{:+.2f}%', 'F1-Score (%)': '{:+.2f}%',
    'Training Time (s)': '{:+.2f}s'
}).background_gradient(cmap='RdYlGn', subset=['Accuracy (%)', 'F1-Score (%)']))

=== LAUNCHING EXPERIMENT 9: PLA vs MLP A/B PIPELINE ===



=== EXPERIMENT 9 PIPELINE COMPLETE ===


/home/nkandasamy/.local/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


,Architecture,Accuracy (%),Precision (%),Recall (%),F1-Score (%),Training Time (s)
Single-Layer Perceptron (PLA),Single Layer (62 Step Units),+15.98%,+22.84%,+15.98%,+12.68%,+14.99s
Tuned Multilayer Perceptron (MLP),"Input(784) -> (128, 64) -> Output(62)",+24.49%,+24.04%,+24.49%,+22.16%,+5.57s
Gain (Delta),MLP Capacity Gain,+8.51%,+1.20%,+8.51%,+9.48%,-9.42s
